In [2]:
# Imports
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import sys
sys.path.append('..')
import prompts
import utils

In [ ]:
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI()

openai_model = os.getenv("OPENAI_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("BASIC_OUTPUT_PATH")

In [4]:
def gpt_nutritionist(image):
    try:
        response = openai_client.chat.completions.create(
            model=openai_model,
            messages=[
            {
                "role": "system",
                "content": prompts.SYSTEM_PROMPT_NUTRITIONIST
            },  
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompts.UNIFIED_NUTRITION_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image}"
                        }
                    }
                ]
            }],
            max_tokens=1024,
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        try:
            data = utils.parse_json(response.choices[0].message.content)
        except Exception as e:
            return {'success': False, 'error': f"No valid JSON found in the response text: {str(e)}"}

        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
def analyze_image_chained(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)

    with open(image_path, "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")

    print(f"\n🔄 Processing {index}: {file_name}")

    print("  Nutrition analysis (GPT)...")
    response = gpt_nutritionist(image_base64)

    if not response.get('success'):
        print(f"❌ Step 2 failed: {response}")
        return {'success': False, 'error': f"Step 2 failed: {response.get('error', 'Unknown error')}", 'index': index}

    print(f"    → {response['description']}")

    total_time = time.time() - start_time
    time.sleep(2)
    print(f"  ✅ Complete! {response['description']} in {total_time:.1f}s")

    return {
        'success': True,
        'id': index,
        'file_name': file_name,
        'response': response,
        'processing_time': total_time
    }

In [6]:
# Process dataset
def process_chained_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""
    
    total_images = utils.count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image_chained(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [7]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['response']
            final_results.append({
                'id': item['index'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [8]:
file_name = "nutria_gpt"

In [9]:
results = process_chained_dataset(file_name, images_folder_path, start=1, end=utils.count_images(images_folder_path))
# results = process_chained_dataset(file_name, images_folder_path, start=1, end=2)

🚀 Processing images 1 to 50 (50 total)

🔄 Processing 1: 1.jpg
  Nutrition analysis (GPT)...
    → Vaso de cerveza rubia clara
  ✅ Complete! Vaso de cerveza rubia clara kcal in 4.2s

🔄 Processing 2: 2.jpg
  Nutrition analysis (GPT)...
    → Pasta tipo penne con salsa de tomate y albahaca
  ✅ Complete! Pasta tipo penne con salsa de tomate y albahaca kcal in 3.4s

🔄 Processing 3: 3.jpg
  Nutrition analysis (GPT)...
    → Croissant integral con semillas, huevo revuelto, lonjas de salmón ahumado y dos tomates cherry
  ✅ Complete! Croissant integral con semillas, huevo revuelto, lonjas de salmón ahumado y dos tomates cherry kcal in 2.4s

🔄 Processing 4: 4.jpg
  Nutrition analysis (GPT)...
    → Dos rebanadas de carne de cerdo con salsa marrón, acompañadas de trozos de tocino frito y papas fritas.
  ✅ Complete! Dos rebanadas de carne de cerdo con salsa marrón, acompañadas de trozos de tocino frito y papas fritas. kcal in 3.8s

🔄 Processing 5: 5.jpg
  Nutrition analysis (GPT)...
    → Porción 

In [10]:
excel_path = export_to_excel(file_name)
print(f"\n🎯 Done! Check: {excel_path}")

✅ Excel exported: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_gpt.xlsx
📊 50 successful analyses

🎯 Done! Check: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_gpt.xlsx
